<h1><b>Modul Pembelajaran Local Search & Optimization RKA 25</b></h1>

<h2><b>Environment Setup</b></h2>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math
import random

<h2><b>1. Penjelasan Local Search dan Optimization</b></h2>

**Apa itu Local Search?**

Local Search adalah kelompok algoritma pencarian yang beroperasi pada **satu state** (atau sekumpulan state) saat ini, 
kemudian bergerak ke state tetangga (*neighbor*) tanpa perlu menyimpan seluruh pohon pencarian. 
Berbeda dengan Informed/Uninformed Search pada modul sebelumnya yang mencari **jalur** dari start ke goal, 
Local Search hanya fokus pada **state akhir yang optimal**.

**Apa itu Optimization?**

Optimization adalah proses menemukan solusi terbaik (global maximum atau global minimum) dari suatu ruang pencarian (*search space*). 
Dalam konteks AI, optimization sering digunakan untuk:
- Tuning hyperparameter model
- Penjadwalan & alokasi sumber daya
- Pathfinding & routing
- Constraint satisfaction

**Konsep Penting:**

| Konsep | Penjelasan |
|---|---|
| Objective / Fitness Function | Fungsi yang mengukur kualitas suatu solusi |
| Local Optimum | Solusi terbaik di antara tetangga-tetangganya, tapi belum tentu terbaik secara keseluruhan |
| Global Optimum | Solusi terbaik dari seluruh ruang pencarian |
| State-Space Landscape | Representasi visual semua state beserta nilai objective function-nya |

<h2><b>2. Hill Climbing</b></h2>

<h3><b>2.1 Definisi</b></h3>

Hill Climbing adalah algoritma pencarian heuristik dari keluarga Local Search. Algoritma ini bekerja dengan cara:
- Secara iteratif berpindah dari state saat ini ke state tetangga yang **lebih baik** menurut fungsi evaluasi
- Bersifat **greedy** — selalu memilih langkah yang memberikan perbaikan langsung
- Merupakan varian dari *generate-and-test algorithm*
- Tidak memerlukan penyimpanan seluruh pohon pencarian

<h3><b>2.2 Jenis-jenis Hill Climbing</b></h3>

1. **Simple Hill Climbing**: Evaluasi tetangga satu per satu, pilih **yang pertama** lebih baik dari state saat ini.
2. **Steepest-Ascent Hill Climbing**: Evaluasi **semua** tetangga, pilih yang memberikan peningkatan **terbesar**.
3. **Stochastic Hill Climbing**: Pilih tetangga secara **random**, terima jika lebih baik. Menambahkan unsur keacakan dalam pemilihan.

<h3><b>2.3 State-Space Diagram</b></h3>

State-Space Diagram adalah representasi visual dari semua state yang mungkin dicapai algoritma:
- **Sumbu-X**: State space (semua konfigurasi yang mungkin)
- **Sumbu-Y**: Nilai objective function

Konsep penting dalam diagram ini:

| Konsep | Penjelasan |
|---|---|
| Local Maximum | State yang lebih baik dari tetangganya, tapi bukan yang terbaik secara keseluruhan |
| Global Maximum | State terbaik di seluruh state-space — solusi optimal |
| Plateau | Daerah datar di mana tetangga-tetangga memiliki nilai yang sama |
| Ridge | Daerah tinggi dengan kemiringan yang bisa terlihat seperti puncak |
| Shoulder | Plateau yang memiliki sisi menanjak — masih bisa naik jika ditelusuri |

<h3><b>2.4 Langkah-langkah</b></h3>

1. **State Awal**: Mulai dari solusi awal (bisa random)
2. **Generate Neighbor**: Buat solusi tetangga dengan melakukan perturbasi kecil pada state saat ini
3. **Evaluasi**: Hitung nilai objective function untuk setiap tetangga
4. **Pindah**: Jika ada tetangga yang lebih baik → pindah ke state tersebut
5. **Terminasi**: Ulangi hingga tidak ada tetangga yang lebih baik (konvergensi)

<h3><b>2.5 Kelebihan dan Kekurangan</b></h3>

**Kelebihan:**
- Sederhana dan mudah diimplementasikan
- Efisien dalam menemukan local optima
- Dapat diterapkan pada berbagai jenis masalah

**Kekurangan:**
- Dapat terjebak di **local optima** (bukan solusi terbaik global)
- Kesulitan di daerah **plateau** (tidak tahu arah mana yang lebih baik)
- Tidak menjamin solusi optimal

<h3><b>2.6 Implementasi</b></h3>

**Step 1:** Definisikan objective function dan fungsi pembangkit tetangga

Kita gunakan fungsi sederhana: $$f(x) = -x^2 + 5 \cos(2 \pi x)$$ yang memiliki minimum di $x = 0$ dengan nilai $f(0) = 0$.

In [ ]:
def objective(x):
    return -(x**2) + 5 * np.cos(2 * np.pi * x)

def generate_neighbors(x, step_size=0.1):
    return [np.array([x[0] + step_size]), np.array([x[0] - step_size])]

**Step 2:** Implementasi algoritma Hill Climbing

In [ ]:
def hill_climbing(objective, initial, n_iterations=100, step_size=0.1):
    current = np.array([initial])
    
    current_eval = objective(current).item() 
    history = [(current[0], current_eval)]

    for i in range(n_iterations):
        neighbors = generate_neighbors(current, step_size)
        
        neighbor_evals = [objective(n).item() for n in neighbors] 
        best_idx = np.argmax(neighbor_evals) 

        if neighbor_evals[best_idx] > current_eval: 
            current = neighbors[best_idx]
            current_eval = neighbor_evals[best_idx]
            
            history.append((current[0], current_eval))
            print(f"Step {i+1}: x = {current[0]:.4f}, f(x) = {current_eval:.4f}")
        else:
            print("Tidak ada tetangga yang lebih baik. Algoritma konvergen.")
            break

    return current, current_eval, history

**Step 3:** Jalankan algoritma

In [ ]:
initial_guess = 2.0
solution, value, history = hill_climbing(objective, initial_guess, n_iterations=100, step_size=0.1)
print(f"\nSolusi terbaik: x = {solution[0]:.4f}, f(x) = {value:.4f}")

**Step 4:** Visualisasi

In [ ]:
x_range = np.linspace(-4, 4, 200)
# Mengganti rumus y_range agar persis dengan objective(x)
y_range = [-(x**2) + 5 * np.cos(2 * np.pi * x) for x in x_range]

hx = [h[0] for h in history]
hy = [h[1] for h in history]

plt.figure(figsize=(10, 5))
# Mengganti label agar sesuai dengan fungsi yang baru
plt.plot(x_range, y_range, 'b-', label='f(x) = -x^2 + 5 cos(2pi x)')
plt.plot(hx, hy, 'ro-', label='Jejak Hill Climbing', markersize=6)
plt.plot(hx[0], hy[0], 'gs', markersize=12, label='Start')
plt.plot(hx[-1], hy[-1], 'r*', markersize=15, label='Solusi')
plt.xlabel('x')
plt.ylabel('f(x)')
# Mengganti judul grafik
plt.title('Hill Climbing pada f(x) = -x^2 + 5 cos(2pi x)')
plt.legend()
plt.grid(True)
plt.show()

<h2><b>3. Simulated Annealing</b></h2>

<h3><b>3.1 Definisi</b></h3>

Simulated Annealing (SA) adalah teknik optimization probabilistik yang terinspirasi dari proses **annealing** dalam metalurgi, 
yaitu proses pemanasan logam kemudian pendinginan secara perlahan untuk menghilangkan cacat dan mencapai struktur kristal yang stabil.

Keunggulan utama SA dibanding Hill Climbing:
- Mampu **keluar dari local optima** melalui probabilitas penerimaan solusi yang lebih buruk
- Probabilitas ini berkurang seiring penurunan "suhu" (*temperature*)
- Pada suhu tinggi → eksplorasi luas; pada suhu rendah → eksploitasi lokal

<h3><b>3.2 Langkah-langkah</b></h3>

1. **Inisialisasi**: Mulai dengan solusi awal $S_0$ dan suhu awal $T_0$
2. **Neighborhood Search**: Generate solusi baru $S'$ dengan perturbasi kecil pada $S$
3. **Evaluasi**: Hitung $\Delta E = f(S') - f(S)$
4. **Acceptance Rule**:
   - Jika $S'$ lebih baik ($\Delta E < 0$ untuk minimization) → **terima**
   - Jika $S'$ lebih buruk → terima dengan probabilitas $P = e^{-\Delta E / T}$
5. **Cooling Schedule**: Kurangi suhu $T$ setiap iterasi: $T_{baru} = \alpha \cdot T$ (exponential cooling)
6. **Terminasi**: Suhu sangat rendah atau iterasi habis

<h3><b>3.3 Cooling Schedule</b></h3>

Cooling schedule sangat mempengaruhi performa SA:
- Terlalu cepat → *premature convergence* (terjebak di local optimum)
- Terlalu lambat → waktu komputasi sangat lama

Contoh cooling schedule:
- **Exponential**: $T_{baru} = \alpha \cdot T$, dengan $\alpha \in [0.8, 0.99]$
- **Linear**: $T_{baru} = T - \Delta T$
- **Logarithmic**: $T_{baru} = \frac{T_0}{\ln(1 + t)}$

<h3><b>3.4 Kelebihan dan Kekurangan</b></h3>

**Kelebihan:**
- Mampu keluar dari local minima/maxima
- Implementasi sederhana
- Mendekati global optimum dengan cooling schedule yang tepat
- Fleksibel — bisa untuk masalah kontinu maupun diskrit

**Kekurangan:**
- Sensitif terhadap parameter (suhu awal, cooling rate)
- Waktu komputasi bisa lama
- Konvergensi lebih lambat dibanding metode deterministik

<h3><b>3.5 Perbandingan SA vs Hill Climbing</b></h3>

| Aspek | Hill Climbing | Simulated Annealing |
|---|---|---|
| Escape local optima | ✗ Tidak bisa | ✓ Bisa (probabilistik) |
| Penerimaan solusi buruk | Tidak pernah | Ya, dengan probabilitas $e^{-\Delta E / T}$ |
| Kecepatan | Cepat | Lebih lambat |
| Hasil | Local optimum | Mendekati global optimum |

<h3><b>3.6 Implementasi</b></h3>

**Step 1:** Definisikan objective function (sama dengan Hill Climbing untuk perbandingan)

In [ ]:
def objective_sa(x):
    return -(x**2) + 5 * np.cos(2 * np.pi * x)

**Step 2:** Implementasi Simulated Annealing

In [ ]:
def simulated_annealing(objective, initial, T=100, alpha=0.95, n_iterations=1000, step_size=0.5):
    current = initial
    current_eval = objective(current)
    best = current
    best_eval = current_eval
    history = [(current, current_eval, T)]

    for i in range(n_iterations):
        # Generate tetangga random
        neighbor = current + random.uniform(-step_size, step_size)
        neighbor_eval = objective(neighbor)

        # Hitung delta
        delta = neighbor_eval - current_eval

        # Acceptance rule untuk MAXIMIZATION
        if delta > 0: # UBAH DI SINI: Jika nilai baru lebih BESAR, terima
            current = neighbor
            current_eval = neighbor_eval
            if current_eval > best_eval: # UBAH DI SINI: Cari best_eval terbesar
                best = current
                best_eval = current_eval
        else:
            # Probabilitas menerima solusi yang lebih buruk (delta bernilai negatif)
            prob = math.exp(delta / T) # TETAP: delta sudah negatif, jadi delta/T aman
            if random.random() < prob:
                current = neighbor
                current_eval = neighbor_eval

        # Cooling schedule
        T = T * alpha
        history.append((current, current_eval, T))

        print(f"Step {i+1}: x = {current:.4f}, f(x) = {current_eval:.4f}")

        if T < 1e-10:
            break

    return best, best_eval, history

**Step 3:** Jalankan dan tampilkan hasil

In [ ]:
best, best_eval, history_sa = simulated_annealing(objective_sa, initial=2.0)
print(f"Solusi terbaik: x = {best:.4f}, f(x) = {best_eval:.4f}")

**Step 4:** Visualisasi perbandingan konvergensi SA

In [ ]:
iters = range(len(history_sa))
vals = [h[1] for h in history_sa]

# Ekstraksi suhu dari history, dengan rumus fallback jika data tidak tersedia
# Pastikan suhu awal (100) dan laju (0.95) cocok dengan kodemu yang sebenarnya
temps = [h[2] if len(h) > 2 else 100 * (0.95 ** i) for i, h in enumerate(history_sa)]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Konfigurasi Sumbu Y Kiri untuk f(x)
color_fx = 'tab:blue'
ax1.set_xlabel('Iterasi')
ax1.set_ylabel('f(x)', color=color_fx)
ax1.plot(iters, vals, color=color_fx, alpha=0.6, label='Nilai f(x) per iterasi')
ax1.axhline(y=best_eval, color='r', linestyle='--', label=f'Terbaik: f(x)={best_eval:.4f}')
ax1.tick_params(axis='y', labelcolor=color_fx)
ax1.grid(True)

# Konfigurasi Sumbu Y Kanan untuk Suhu
ax2 = ax1.twinx()  
color_temp = 'tab:red'
ax2.set_ylabel('Suhu (T)', color=color_temp)  
ax2.plot(iters, temps, color=color_temp, alpha=0.8, label='Penurunan Suhu')
ax2.tick_params(axis='y', labelcolor=color_temp)

fig.tight_layout()  
plt.title('Konvergensi Simulated Annealing dan Perubahan Suhu')

# Menggabungkan legenda dari kedua sumbu agar rapi
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='lower right')

plt.show()

<h2><b>4. Genetic Algorithm</b></h2>

<h3><b>4.1 Definisi</b></h3>

Genetic Algorithm (GA) adalah teknik optimization berbasis **populasi** yang terinspirasi dari prinsip **seleksi alam** dan **genetika**. 
Berbeda dengan Hill Climbing dan Simulated Annealing yang bekerja pada satu solusi, GA mengevolusi **populasi** solusi secara bersamaan.

Karakteristik utama:
- Bekerja pada banyak solusi sekaligus (population-based)
- Menggunakan operator biologis: **selection**, **crossover**, **mutation**
- Efektif untuk masalah non-linear, non-convex, dan multimodal
- Tidak memerlukan informasi gradient

<h3><b>4.2 Konsep Utama</b></h3>

| Konsep | Penjelasan |
|---|---|
| **Population** | Kumpulan candidate solutions pada satu generasi |
| **Chromosome** | Satu candidate solution lengkap |
| **Gene** | Unit terkecil dalam chromosome, merepresentasikan satu variabel |
| **Fitness Function** | Fungsi yang mengevaluasi kualitas chromosome |

**Metode Encoding:**
- **Binary Encoding**: Menggunakan string biner (0 dan 1)
- **Real-Valued Encoding**: Gene berupa bilangan real — cocok untuk optimization kontinu
- **Permutation Encoding**: Chromosome merepresentasikan urutan — cocok untuk TSP, scheduling

**Kriteria Terminasi:**
- Jumlah generasi maksimum tercapai
- Fitness threshold tercapai
- Tidak ada perbaikan selama beberapa generasi
- Batas waktu komputasi

<h3><b>4.3 Genetic Operator</b></h3>

**a. Selection** — Memilih chromosome untuk menjadi parent:
- **Roulette Wheel Selection**: Probabilitas seleksi proporsional terhadap fitness
- **Tournament Selection**: Pilih acak beberapa individu, ambil yang terbaik
- **Stochastic Universal Sampling (SUS)**: Versi perbaikan roulette wheel dengan multiple pointer

**b. Crossover** — Menggabungkan genetic material dari dua parent:
- **One-Point Crossover**: Satu titik potong random
- **Multi-Point Crossover**: Dua titik potong random
- **Davis Order Crossover (OX1)**: Untuk permutation encoding
- **Uniform Crossover**: Tiap gene dipilih random dari salah satu parent

**c. Mutation** — Perubahan random pada gene:
- **Bit Flip**: Membalik bit (untuk binary encoding)
- **Swap**: Menukar posisi dua gene
- **Scramble**: Mengacak segmen dalam chromosome
- **Inversion**: Membalik urutan segmen

<h3><b>4.4 Langkah-langkah</b></h3>

1. **Inisialisasi Populasi**: Generate populasi awal secara random
2. **Evaluasi Fitness**: Hitung fitness setiap chromosome
3. **Seleksi Parent**: Pilih parent berdasarkan fitness
4. **Crossover**: Gabungkan parent untuk menghasilkan offspring
5. **Mutasi**: Terapkan perubahan random pada offspring
6. **Pembentukan Generasi Baru**: Ganti populasi lama dengan offspring baru
7. **Cek Terminasi**: Periksa apakah kriteria berhenti terpenuhi
8. **Output**: Kembalikan chromosome terbaik

<h3><b>4.5 Kelebihan dan Kekurangan</b></h3>

**Kelebihan:**
- Efektif untuk masalah non-linear dan multimodal
- Tidak memerlukan gradient — bisa untuk fungsi yang tidak differentiable
- Population-based → eksplorasi ruang pencarian lebih luas
- Fleksibel dalam representasi masalah

**Kekurangan:**
- Komputasi lebih mahal (banyak evaluasi fitness per generasi)
- Banyak parameter yang harus di-tuning (population size, mutation rate, crossover rate)
- Tidak menjamin menemukan global optimum

<h3><b>4.6 Implementasi</b></h3>

**Step 1:** Definisikan fitness function

Kita gunakan fungsi multimodal $f(x) = x \cdot \sin(10\pi x) + 1$ untuk mendemonstrasikan kemampuan GA menghindari local optima.

In [ ]:
def fitness_function(x):
    return x * np.sin(10 * np.pi * x) + 1

**Step 2:** Definisikan parameter dan inisialisasi populasi

In [ ]:
POP_SIZE = 40
GENERATIONS = 1000
X_MIN, X_MAX = -1.0, 2.0
CROSSOVER_PROB = 0.9
MUTATION_PROB = 0.2
MUTATION_STD = 0.1

np.random.seed(42)
population = np.random.uniform(X_MIN, X_MAX, POP_SIZE)

**Step 3:** Implementasi genetix operator

In [ ]:
def tournament_selection(pop, fitness, k=3):
    selected = []
    for _ in range(len(pop)):
        idx = np.random.choice(len(pop), k, replace=False)
        selected.append(pop[idx[np.argmax(fitness[idx])]])
    return np.array(selected)

def arithmetic_crossover(p1, p2):
    alpha = np.random.rand()
    return alpha * p1 + (1 - alpha) * p2, alpha * p2 + (1 - alpha) * p1

def mutate(x):
    if np.random.rand() < MUTATION_PROB:
        x += np.random.normal(0, MUTATION_STD)
    return np.clip(x, X_MIN, X_MAX)

**Step 4:** Loop evolusi

In [ ]:
best_history = []
mean_history = []

for gen in range(GENERATIONS):
    fitness = fitness_function(population)
    best_history.append(np.max(fitness))
    mean_history.append(np.mean(fitness))

    # Mencari indeks individu terbaik di generasi ini untuk dicetak
    best_idx_gen = np.argmax(fitness)
    print(f"Step {gen+1}: x terbaik = {population[best_idx_gen]:.4f}, f(x) = {fitness[best_idx_gen]:.4f}")

    parents = tournament_selection(population, fitness)
    offspring = []
    np.random.shuffle(parents)

    for i in range(0, POP_SIZE, 2):
        if np.random.rand() < CROSSOVER_PROB:
            c1, c2 = arithmetic_crossover(parents[i], parents[i + 1])
        else:
            c1, c2 = parents[i], parents[i + 1]
        offspring.extend([mutate(c1), mutate(c2)])

    population = np.array(offspring)

# Hasil akhir
final_fitness = fitness_function(population)
best_idx = np.argmax(final_fitness)
print(f"Solusi terbaik: x = {population[best_idx]:.4f}, f(x) = {final_fitness[best_idx]:.4f}")

**Step 5:** Visualisasi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Konvergensi fitness
axes[0].plot(best_history, label='Best Fitness')
axes[0].plot(mean_history, label='Mean Fitness')
axes[0].set_xlabel('Generasi')
axes[0].set_ylabel('Fitness')
axes[0].set_title('Konvergensi GA')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Populasi akhir pada fungsi objektif
x = np.linspace(X_MIN, X_MAX, 500)
axes[1].plot(x, fitness_function(x), 'b-', label='f(x)')
axes[1].scatter(population, fitness_function(population), c='red', zorder=5, label='Populasi Akhir')
axes[1].scatter(population[best_idx], final_fitness[best_idx], c='gold', s=200, marker='*', zorder=6, label='Terbaik')
axes[1].set_xlabel('x')
axes[1].set_ylabel('f(x)')
axes[1].set_title('Populasi Akhir pada Objective Function')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

<h2><b>5. Perbandingan Ketiga Algoritma</b></h2>

<h3><b>5.1 Tabel Perbandingan</b></h3>

| Aspek | Hill Climbing | Simulated Annealing | Genetic Algorithm |
|---|---|---|---|
| Tipe | Single-state | Single-state | Population-based |
| Escape Local Optima | ✗ | ✓ (probabilistik) | ✓ (populasi + crossover) |
| Parameter Tuning | Minimal | Moderate (T, α) | Kompleks (pop, mutation, crossover) |
| Kecepatan | Cepat | Moderate | Lambat |
| Memori | O(1) | O(1) | O(pop_size) |
| Kualitas Solusi | Local optimum | Mendekati global | Mendekati global |

<h3><b>5.2 Kapan Menggunakan Local Search</b></h3>

Gunakan Local Search ketika:
- Kombinasi kemungkinan solusi terlalu masif
- Hanya membutuhkan hasil akhir optimal
- Kapasitas memori perangkat sangat terbatas

Contoh aplikasi dunia nyata:
- Sistem Navigasi: Mencari rute pengiriman paling optimal
- Sistem Penjadwalan: Menyusun kelas tanpa jadwal bentrok

<h2><b>6. Hands-on</b></h2>

Tujuan
- Mengimplementasikan Hill Climbing
- Mengimplementasikan Simulated Annealing
- Mengimplementasikan Genetic Algorithm

Instruksi:
- Lengkapi bagian kode yang kosong (_____)

<font color="blue">Latihan 1</font>

N-Queens Problem — Tempatkan N ratu pada papan N×N sehingga tidak ada yang saling menyerang. 
Gunakan Hill Climbing di mana: state = konfigurasi ratu (satu per kolom), neighbor = pindahkan 1 ratu dalam kolom, 
fitness = jumlah pasangan ratu yang TIDAK saling menyerang.

In [ ]:
def count_non_attacking(queens):
    """Hitung jumlah pasangan ratu yang tidak saling menyerang."""
    n = len(queens)
    non_attacking = 0
    for i in range(n):
        for j in range(i+1, n):
            if queens[i] != queens[j] and abs(queens[i] - queens[j]) != abs(i - j):
                # Cek bukan di diagonal yang sama
                non_attacking += 1
    return non_attacking

def get_neighbors_queens(queens):
    """Generate semua neighbor: pindahkan 1 ratu ke baris lain di kolomnya."""
    neighbors = []
    n = len(queens)
    for col in range(n):
        for row in range(n):
            if queens[col] != row:
                new_queens = list(queens)
                new_queens[col] = row
                neighbors.append(tuple(new_queens))
    return neighbors

def hill_climbing_queens(n=8, max_iter=1000):
    current = tuple(random.randint(0, n-1) for _ in range(n))
    current_fitness = count_non_attacking(current)
    max_fitness = n * (n - 1) // 2  # semua pasangan tidak menyerang

    for i in range(max_iter):
        if current_fitness == max_fitness:
            return current, current_fitness, True

        neighbors = get_neighbors_queens(current)
        best = max(neighbors, key=lambda q: count_non_attacking(q))
        best_fitness = count_non_attacking(best)

        if best_fitness <= current_fitness:
            return current, current_fitness, False  # terjebak

        current = best
        current_fitness = best_fitness

    return current, current_fitness, False

In [ ]:
sol, fit, success = hill_climbing_queens(8)
print(f"Solusi: {sol}, Fitness: {fit}, Berhasil: {success}")

<font color="blue">Latihan 2</font>

Gunakan SA untuk memecahkan N-Queens Problem (N=8). 
Bandingkan success rate SA vs Hill Climbing dari 20 percobaan.

In [ ]:
def sa_queens(n=8, T=4.0, alpha=0.99, n_iter=5000):
    current = list(range(n))
    random.shuffle(current)
    current_fitness = count_non_attacking(current)
    max_fitness = n * (n-1) // 2

    for i in range(n_iter):
        if current_fitness == max_fitness:
            return tuple(current), current_fitness, True

        new = current[:]
        col = random.randint(0, n-1)
        new[col] = random.randint(0, n-1)  # random row baru
        new_fitness = count_non_attacking(new)

        delta = new_fitness - current_fitness
        if delta > 0 or (T > 0 and random.random() < math.exp(delta / T)):
            current = new
            current_fitness = new_fitness

        T *= alpha

    return tuple(current), current_fitness, current_fitness == max_fitness

In [ ]:
# Bandingkan SA vs HC
sa_success = sum(sa_queens()[2] for _ in range(20))
hc_success = sum(hill_climbing_queens()[2] for _ in range(20))
print(f"SA: {sa_success}/20, HC: {hc_success}/20")

<font color="blue">Latihan 3</font>

Implementasikan GA untuk TSP sederhana (6 kota). 
Gunakan permutation encoding, order crossover (OX1), dan swap mutation.

In [ ]:
def order_crossover(p1, p2):
    n = len(p1)
    i, j = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[i:j+1] = p1[i:j+1]   # segment dari p1
    fill = [g for g in p2 if g not in p1[i:j+1]]
    idx = 0
    for k in range(n):
        if child[k] is None:
            child[k] = fill[idx]
            idx += 1
    return child

def swap_mutation(route, prob=0.1):
    if random.random() < prob:
        i, j = random.sample(range(len(route)), 2)
        route[i], route[j] = route[j], route[i]
    return route

def ga_tsp(cities, pop_size=50, gens=200):
    names = list(cities.keys())
    pop = [random.sample(names, len(names)) for _ in range(pop_size)]

    for gen in range(gens):
        fitnesses = [1 / total_distance(r, cities) for r in pop]
        new_pop = []
        for _ in range(pop_size // 2):
            p1 = max(random.sample(pop, 3), key=lambda r: 1/total_distance(r, cities))
            p2 = max(random.sample(pop, 3), key=lambda r: 1/total_distance(r, cities))
            c1 = order_crossover(p1, p2)
            c2 = order_crossover(p1,p2)
            new_pop.extend([swap_mutation(c1), swap_mutation(c2)])
        pop = new_pop

    best = min(pop, key=lambda r: total_distance(r, cities))
    return best, total_distance(best, cities)

In [ ]:
route, dist = ga_tsp(cities)
print(f"GA TSP Route: {route}, Distance: {dist:.2f}")